In [1]:
import os, cv2, torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler

# =====================
# CONFIG
# =====================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CSV_PATH = "/kaggle/input/rsna-bone-age/boneage-training-dataset.csv"
PATCH_ROOT = "/kaggle/input/phase-1"

HAND_SIZE = 299
PATCH_SIZE = 224

BATCH_SIZE = 16
EPOCHS = 50
LR = 2e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 8

# =====================
# VERIFY PATCHES
# =====================
def verify_folders():
    print("Verifying patch folders...")
    for d in ["hand_patches", "region1_patches", "region2_patches"]:
        path = os.path.join(PATCH_ROOT, d)
        if not os.path.exists(path):
            raise RuntimeError(f"Missing folder: {path}")
        print(f"{d}: {len(os.listdir(path))} files")
    print("All patch folders OK.\n")

# =====================
# DATASET
# =====================
class PatchDataset(Dataset):
    def __init__(self, df, train=True):
        self.df = df.reset_index(drop=True)

        aug = transforms.RandomHorizontalFlip() if train else transforms.Lambda(lambda x:x)

        self.tf_hand = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((HAND_SIZE, HAND_SIZE)),
            aug,
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
        ])

        self.tf_patch = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((PATCH_SIZE, PATCH_SIZE)),
            aug,
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
        ])

    def load_img(self, folder, img_id, is_hand=False):
        path = f"{PATCH_ROOT}/{folder}/{img_id}.jpg"
        img = cv2.imread(path)
        if img is None:
            raise FileNotFoundError(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return self.tf_hand(img) if is_hand else self.tf_patch(img)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        img_id = int(r.id)

        hand = self.load_img("hand_patches", img_id, True)
        r1   = self.load_img("region1_patches", img_id)
        r2   = self.load_img("region2_patches", img_id)

        gender = torch.tensor(r.male, dtype=torch.float32)
        age    = torch.tensor(r.boneage, dtype=torch.float32)

        return hand, r1, r2, gender, age

    def __len__(self):
        return len(self.df)

# =====================
# MODEL
# =====================
class Stage2PatchFusionNet(nn.Module):
    def __init__(self):
        super().__init__()

        base = models.inception_v3(
            weights=models.Inception_V3_Weights.IMAGENET1K_V1,
            aux_logits=True
        )

        # ---- correct feature extractor ----
        self.hand_net = nn.Sequential(
            base.Conv2d_1a_3x3,
            base.Conv2d_2a_3x3,
            base.Conv2d_2b_3x3,
            base.maxpool1,
            base.Conv2d_3b_1x1,
            base.Conv2d_4a_3x3,
            base.maxpool2,
            base.Mixed_5b,
            base.Mixed_5c,
            base.Mixed_5d,
            base.Mixed_6a,
            base.Mixed_6b,
            base.Mixed_6c,
            base.Mixed_6d,
            base.Mixed_6e,
            base.Mixed_7a,
            base.Mixed_7b,
            base.Mixed_7c,
        )

        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        self.r1_net = nn.Sequential(*list(resnet.children())[:-1])

        vgg = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1)
        self.r2_net = nn.Sequential(*list(vgg.features), nn.AdaptiveAvgPool2d(1))

        self.gender_fc = nn.Linear(1, 32)

        self.regressor = nn.Sequential(
            nn.Linear(2048 + 2048 + 512 + 32, 1024),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(1024, 1)
        )

    def forward(self, h, r1, r2, g):
        f1 = self.hand_net(h).mean(dim=[2,3])   # GAP
        f2 = self.r1_net(r1).flatten(1)
        f3 = self.r2_net(r2).flatten(1)
        g  = self.gender_fc(g.unsqueeze(1))

        return self.regressor(torch.cat([f1,f2,f3,g],1)).squeeze(1)


# =====================
# TRAINING
# =====================
def train():
    verify_folders()

    df = pd.read_csv(CSV_PATH)
    valid_ids = set(int(f[:-4]) for f in os.listdir(f"{PATCH_ROOT}/hand_patches"))
    df = df[df["id"].isin(valid_ids)].reset_index(drop=True)

    train_df = df.sample(frac=0.9, random_state=42)
    val_df   = df.drop(train_df.index)

    train_dl = DataLoader(PatchDataset(train_df, True), BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_dl   = DataLoader(PatchDataset(val_df, False), BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    model = Stage2PatchFusionNet().to(DEVICE)

    opt = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = GradScaler()

    best = 1e9
    bad = 0

    for ep in range(EPOCHS):
        model.train()
        tr_loss = 0

        for h,r1,r2,g,a in tqdm(train_dl, desc=f"Epoch {ep+1}"):
            h,r1,r2,g,a = [x.to(DEVICE) for x in (h,r1,r2,g,a)]

            with autocast():
                pred = model(h,r1,r2,g)
                loss = torch.mean(torch.abs(pred - a))

            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

            tr_loss += loss.item()

        tr_loss /= len(train_dl)

        model.eval()
        va_loss = 0
        with torch.no_grad():
            for h,r1,r2,g,a in val_dl:
                h,r1,r2,g,a = [x.to(DEVICE) for x in (h,r1,r2,g,a)]
                va_loss += torch.mean(torch.abs(model(h,r1,r2,g) - a)).item()

        va_loss /= len(val_dl)

        print(f"Epoch {ep+1}: Train MAE {tr_loss:.3f} | Val MAE {va_loss:.3f}")

        if va_loss < best:
            best = va_loss
            bad = 0
            torch.save(model.state_dict(), "stage2_best.pt")
            print("✅ Saved best model")
        else:
            bad += 1
            if bad >= PATIENCE:
                print("⏹ Early stopping")
                break

    print("\nBest Validation MAE:", best)

# =====================
if __name__ == "__main__":
    train()


Verifying patch folders...
hand_patches: 12581 files
region1_patches: 12581 files
region2_patches: 12581 files
All patch folders OK.

Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth


100%|██████████| 104M/104M [00:00<00:00, 230MB/s] 


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 191MB/s]


Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


...

Epoch 27: Train MAE 7.542 | Val MAE 9.036


Epoch 28: 100%|██████████| 708/708 [04:36<00:00,  2.56it/s]


Epoch 28: Train MAE 7.404 | Val MAE 9.064


Epoch 29: 100%|██████████| 708/708 [04:36<00:00,  2.56it/s]


Epoch 29: Train MAE 7.366 | Val MAE 9.102
⏹ Early stopping

Best Validation MAE: 8.765194663518592
